# Inspect direct beam histogram with QuantiCam
- [X] Redirect laser elongated profile directly into the sensor to observe sharp peak without internal reflection from the obscure box
- [X] HDF5 data acquisition stream
- [ ] Peak detection algorithm employed
- [ ] Model Poission noise for the direct path

| Layout | Illumination |
|:--------:|:--------|
| ![alt text](./img/PXL_20250317_Direct_Laser_Hist_Layout.jpg "Beam path") | ![alt text](./img/PXL_20250317_Direct_Laser_Hist_DarkIllumination.jpg "Beam path") |  

## Setup connection to the QC FPGA
- Write FW
- Config voltages
- Config sensor settings for the mode we use

In [ ]:
using QuantiCam

#fw_path = "../hw/TOP_7310_modes_1_2.bit"
fw_path = "../hw/photon_cnt_tcspc_xem7310-a200_v1.8.2.bit"

if !(@isdefined qc) || qc === nothing || qc.fpga.bitfile != fw_path
    if (@isdefined qc) && qc!== nothing
        cleanup!(qc)
    end
    qc = QCBoard(fw_path, "../config/tcspc.json")
    init_board!(qc)
else
    # Try to reconfigure or cleanup and initialise from the begining
    try
        reload_config(qc, "../config/tcspc.json")
        config_sensor(qc)
    catch
        cleanup!(qc)
    end
end

[ Info: Opal Kelly to API Comms setup in progress...


Scanning USB for Opal Kelly devices...
Found 1 Opal Kelly device(s)
Serial number of device 0 is 1908000OVV


[ Info: Device opened with id=Opal Kelly XEM7310 and board model=ok_brdXEM7310A200
[ Info: Located bit file ../hw/TOP_7310_modes_1_2.bit
[ Info: Firmware written successfuly
[ Info: Firmware revision: 0.0.0
[ Info: Waiting on voltages to stabilize
[ Info: Connected to Sensor
[ Info: Initialize logic parameters necessary to interact with the sensor
[ Info: Reset sensor and set parameters for the MODE of use
[ Info: Sensor configured


In [ ]:
cleanup!(qc)
qc = nothing

In [ ]:
using QuantiCam
using Statistics
using Plots
plotlyjs()

frames = capture_frames(qc, 1000);
filtered_frames::Vector{Union{UInt16, Missing}} = map(frame -> QuantiCam.filter_code(frame), frames)

function collect_frames(v::Vector{Matrix{T}}) where T
    n_rows = size(v[1], 1)
    n_cols = size(v[1], 2)
    n_matrices = length(v)
    
    result = Matrix{Vector{T}}(undef, n_rows, n_cols)
    
    for i in 1:n_rows, j in 1:n_cols
        result[i, j] = [v[k][i, j] for k in 1:n_matrices]
    end
    
    return result
end

#tcspc_stream = collect_frames([frames[:,:,i] for i in 1:size(frames,3)])
tcspc_stream_missing = collect_frames(filtered_frames)
tcspc_stream = map(pixel -> collect(skipmissing(pixel)), tcspc_stream_missing)
tcspc_mean = map(pixel -> mean(pixel), tcspc_stream)
tcspc_var = map(pixel -> var(pixel), tcspc_stream);

# ND=1.5
all_pixels = collect(Iterators.flatten(tcspc_stream[:,5:128]))
histogram(all_pixels, nbins=4096)

┌ Warning: FRAME(108): Frames skipped expected_idx=108 to new_idx=141 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(144): Frames skipped expected_idx=177 to new_idx=164 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(604): Frames skipped expected_idx=112 to new_idx=114 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75


LoadError: MethodError: no method matching collect_frames(::Vector{Matrix})
The function `collect_frames` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  collect_frames([91m::Array{Matrix{T}, 1}[39m) where T
[0m[90m   @[39m [32mMain[39m [90m[4mIn[5]:9[24m[39m


In [9]:
function collect_frames(v::Vector{Matrix})
    n_rows = size(v[1], 1)
    n_cols = size(v[1], 2)
    n_matrices = length(v)
    
    result = Matrix{Vector}(undef, n_rows, n_cols)
    
    for i in 1:n_rows, j in 1:n_cols
        result[i, j] = [v[k][i, j] for k in 1:n_matrices]
    end
    
    return result
end
tcspc_stream_missing = collect_frames(filtered_frames)

192×128 Matrix{Vector}:
 UInt16[0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f  …  0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f, 0x005f]  …  UInt16[0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004  …  0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004]
 UInt16[0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e  …  0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e, 0x005e]     UInt16[0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004  …  0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004]
 UInt16[0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d  …  0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d, 0x005d]     UInt16[0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004, 0x0004  …  0x0004, 0x0004, 0x000

In [12]:
filtered_frames

1000-element Vector{Matrix}:
 UInt16[0x005f 0x8001 … 0x0004 0x0004; 0x005e 0x8001 … 0x0004 0x0004; … ; 0x0004 0x0004 … 0x0004 0x0004; 0x0004 0x0004 … 0x0004 0x0004]
 Union{Missing, UInt16}[0x005f 0x8002 … 0x0004 0x0004; 0x005e 0x8002 … 0x0004 0x0004; … ; 0x0004 0x0004 … 0x0004 0x0004; 0x0004 0x0004 … 0x0004 0x0004]
 UInt16[0x005f 0x8003 … 0x0004 0x0004; 0x005e 0x8003 … 0x0004 0x0004; … ; 0x0004 0x0e4c … 0x0004 0x0004; 0x0e0c 0x0004 … 0x0004 0x0004]
 UInt16[0x005f 0x8004 … 0x0004 0x0004; 0x005e 0x8004 … 0x0004 0x0004; … ; 0x0004 0x0004 … 0x0004 0x0004; 0x0232 0x0004 … 0x0004 0x0004]
 UInt16[0x005f 0x8005 … 0x0004 0x0004; 0x005e 0x8005 … 0x0004 0x0004; … ; 0x0004 0x0004 … 0x0004 0x0004; 0x0004 0x0004 … 0x0004 0x0004]
 UInt16[0x005f 0x8006 … 0x0004 0x0004; 0x005e 0x8006 … 0x0004 0x0004; … ; 0x0004 0x0e41 … 0x0004 0x0004; 0x08e3 0x0004 … 0x0004 0x0004]
 Union{Missing, UInt16}[0x005f 0x8007 … 0x0004 0x0004; 0x005e 0x8007 … 0x0004 0x0004; … ; 0x0004 0x0004 … 0x0004 0x0004; 0x050a 0x0064 … 0x

In [10]:
using StatsBase
function get_maximum(pixel::Vector{T}; nbins=200) where T
    pixel_stream = collect(skipmissing(pixel))
    # Calculate the bin edges based on the range of data and the number of bins
    bin_edges = range(0, stop=4095, length=nbins+1)
    h = fit(Histogram, pixel_stream, bin_edges)
    # Extract histogram data
    bin_counts = h.weights
    bin_centers = h.edges[1:end-1] .+ diff(collect(h.edges)) ./ 2  # Calculate bin centers

    # Peak extraction: Find the bin with the maximum count
    peak_index = argmax(bin_counts)
    peak_value = bin_centers[peak_index]
    peak_count = bin_counts[peak_index]
    return peak_value
end
get_maximum(all_pixels, nbins=2000)

LoadError: BoundsError: attempt to access 0-element Vector{Union{}} at index [464]

In [47]:
using StatsBase
# Define the number of bins
num_bins = 2048
# Calculate the bin edges based on the range of data and the number of bins
bin_edges = range(0, stop=4095, length=num_bins+1)
h = fit(Histogram, all_pixels, bin_edges)
# Extract histogram data
bin_counts = h.weights
bin_edges_array = collect(h.edges[1])  # Convert edges tuple to array
bin_centers = bin_edges_array[1:end-1] .+ diff(bin_edges_array) ./ 2  # Calculate bin centers
# Peak extraction: Find the bin with the maximum count
peak_index = argmax(bin_counts)
peak_count = bin_counts[peak_index]
peak_value = bin_centers[peak_index]

3750.084228515625

In [37]:
qc.config.header_en

false

In [ ]:
histogram(all_pixels, bins=500)

In [ ]:
capture_frames(qc, 50000);

┌ Warning: FRAME(582): Frames skipped expected_idx=69 to new_idx=98 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(1676): Frames skipped expected_idx=168 to new_idx=108 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(1677): Frames skipped expected_idx=109 to new_idx=112 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(2224): Frames skipped expected_idx=147 to new_idx=149 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(30141): Frames skipped expected_idx=162 to new_idx=163 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75


In [69]:
frames = capture_frames(qc, 5000);
filtered_frames = map(frame -> QuantiCam.filter_code(frame), frames)
tcspc_stream_missing = collect_frames(filtered_frames)
tcspc_stream = map(pixel -> collect(skipmissing(pixel)), tcspc_stream_missing)
tcspc_mean = map(pixel -> mean(pixel), tcspc_stream)
tcspc_var = map(pixel -> var(pixel), tcspc_stream);
histogram(tcspc_stream[70, 44], bins=200)

┌ Warning: Sensor already disconnected!
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/setup.jl:140
┌ Warning: FRAME(3): Frames skipped expected_idx=3 to new_idx=4 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(324): Frames skipped expected_idx=69 to new_idx=102 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(645): Frames skipped expected_idx=167 to new_idx=190 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(966): Frames skipped expected_idx=255 to new_idx=68 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/utils.jl:75
┌ Warning: FRAME(1287): Frames skipped expected_idx=133 to new_idx=224 => Non continuous frames capture
└ @ QuantiCam ~/Documents/Scripts/Julia/opal-kelly/QuantiCam/src/u

LoadError: MethodError: no method matching collect_frames(::Vector{Matrix})
The function `collect_frames` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  collect_frames([91m::Array{Matrix{T}, 1}[39m) where T
[0m[90m   @[39m [36mMain[39m [90m[4mIn[52]:9[24m[39m


## Test if empty read results in a FIFO_EMPTY error

In [ ]:
using Plots
#plotlyjs() # plotlyjs would work in a python env with webio_jupyter_extension package
#one_frame = capture_frame(qc)
one_frame = frames[13]
filtered_frame = QuantiCam.filter_code(one_frame)
heatmap(one_frame[:,5:128])

## TCSPC Histogram
- [ ] Aquire N tcspc frames
- [ ] Make histogram for each pixel

# Interpret

- Read frames from HDF5

In [ ]:
using QuantiCam

filtered_frames = map(frame -> QuantiCam.filter_code(frame), frames)
typeof(filtered_frames)

In [ ]:
using Statistics

function collect_frames(v::Vector{Matrix{T}}) where T
    n_rows = size(v[1], 1)
    n_cols = size(v[1], 2)
    n_matrices = length(v)
    
    result = Matrix{Vector{T}}(undef, n_rows, n_cols)
    
    for i in 1:n_rows, j in 1:n_cols
        result[i, j] = [v[k][i, j] for k in 1:n_matrices]
    end
    
    return result
end

#tcspc_stream = collect_frames([frames[:,:,i] for i in 1:size(frames,3)])
tcspc_stream_missing = collect_frames(filtered_frames)
tcspc_stream = map(pixel -> collect(skipmissing(pixel)), tcspc_stream_missing)
tcspc_mean = map(pixel -> mean(pixel), tcspc_stream)
tcspc_var = map(pixel -> var(pixel), tcspc_stream);

In [ ]:
using Plots
plotlyjs()
frames_center = map(frame -> frame[1:192, 1:123], filtered_frames)
tcspc_events = map(pixel -> length(collect((skipmissing(pixel)))), tcspc_stream)
#heatmap(map(pixel -> length(collect((skipmissing(pixel)))), tcspc_stream))
#heatmap(map(pixel -> sum(collect((skipmissing(pixel)))), collect_frames(frames_center)))
heatmap(tcspc_events[:,5:123])
#heatmap(filtered_frames[24])

In [ ]:
pixel = UInt16.(tcspc_stream[79, 95])
# For each value in the pixel vector, invert the bits
inverted_pixel = map(value -> bitstring(value)[12:-1:1], pixel)
inverted_pixel_uint16 = map(bits -> parse(UInt16, bits, base=2), inverted_pixel)
println(inverted_pixel_uint16)
println(pixel)
println(inverted_pixel)

In [ ]:
histogram(tcspc_stream[90, 72], bins=200)
#histogram(inverted_pixel_uint16, bins=200, xlabel="Inverted TCSPC Value", ylabel="Count", title="Histogram of Inverted TCSPC Values at Pixel (79, 95)")
#map(x -> UInt16(x) & 0xF, tcspc_stream[75, 45])

In [ ]:
using StatsBase
function get_maximum(pixel::Vector{T}) where T
    pixel_stream = collect(skipmissing(pixel))
    h = fit(Histogram, pixel_stream, 0.0:1.0:255.0)
    idx = findfirst(==(maximum(h.weights)), h.weights)
    return h.edges[1][idx]
end
peaks = map(pixel -> get_maximum(pixel), tcspc_stream)
heatmap(peaks[:, 5:127])
# TODO: More useful would be in this case to plot uncertainty
#get_maximum(tcspc_stream[110, 20])

In [ ]:
tcspc_var = map(pixel -> var(pixel[1],mean=pixel[2]), zip(tcspc_stream, peaks))
tcspc_std = map(pixel -> std(pixel[1],mean=pixel[2]), zip(tcspc_stream, peaks))
#typeof(peaks)
heatmap(map(peak -> minimum([peak, 150.0]), peaks[:, 5:127]))

### TDC Clock skew
The STOP clock is skewed lineraly across the sensor along the columns axis.

In [ ]:
tcspc_col_stream = [vcat(tcspc_stream[:, col]...) for col in 1:size(tcspc_stream, 2)]
peaks_col = map(pixel -> get_maximum(pixel), tcspc_col_stream)
typeof(tcspc_col_stream)
plot(5:127, peaks_col[5:127])

In [ ]:
# ND=0.5
all_pixels = collect(Iterators.flatten(tcspc_stream[:,5:128]))
histogram(all_pixels, bins=1000)